# Use imodels 

pip install imodels

['山鸢尾', '变色鸢尾', '维吉尼亚鸢尾']

['setosa' 'versicolor' 'virginica']

['萼片长度（厘米）', '萼片宽度（厘米）', '花瓣长度（厘米）', '花瓣宽度（厘米）']

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load dataset
# https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset 
data = load_iris()
X = data.data
y = data.target

# Filter the dataset to include only two classes (e.g., class 0 and class 1)
binary_class_indices = np.where((y == 0) | (y == 1))
X = X[binary_class_indices]
y = y[binary_class_indices]


In [2]:
print(X.shape)
print(y.shape)
print(data.target_names)
print(data.feature_names)

(100, 4)
(100,)
['setosa' 'versicolor' 'virginica']
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


## Use imodels BayesianRuleListClassifier

The limitation of pre-mined rules are determined by the descretizer and hyperparameter choices, and has nothing to do with the "learning" from data overrall and fail to generate multiple thresholds for each feature. 

BRL by imodels only implement binary classification too. 

In [9]:
from sklearn.preprocessing import KBinsDiscretizer

# Discretize the features
discretizer = KBinsDiscretizer(n_bins=2, encode='ordinal', strategy='uniform')
X_discretized = discretizer.fit_transform(X)
# Convert the discretized data to integer type, as BRL expects categorical data in integer format
X_discretized = X_discretized.astype(int)

In [10]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_discretized, y, test_size=0.3, random_state=42)


In [4]:
from imodels import BayesianRuleListClassifier

In [11]:
# X_discretized[:5]
X[:5]
X_train[:5]
y_train[:5]

array([0, 0, 1, 0, 1])

In [13]:

# Initialize and train the Bayesian Rule List model
brl = BayesianRuleListClassifier()
brl.fit(X_train, y_train, feature_names=data.feature_names)

# Make predictions
y_pred = brl.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

/Users/jingjing/miniconda3/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:109: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Accuracy: 1.0


In [16]:
# Print the learned rule list
print("Learned Rule List:")
brl

Learned Rule List:


Trained RuleListClassifier 
============================
IF petal width (cm) > 0.5 THEN probability of class 1: 97.4% (90.7%-99.9%)
ELSE IF petal length (cm) > 0.5 THEN probability of class 1: 50.0% (2.5%-97.5%)
ELSE probability of class 1: 2.9% (0.1%-10.3%)
===========================

In [18]:
discretizer.bin_edges_ 

array([array([4.3 , 5.65, 7.  ]), array([2. , 3.2, 4.4]),
       array([1.  , 3.05, 5.1 ]), array([0.1 , 0.95, 1.8 ])], dtype=object)

# x - Use sklearn-expertsys
:x: The expertsys is an implementation and does not work 

https://github.com/tmadl/sklearn-expertsys

install pyfim




# Do generation algorithm from scratch 

`pip install mlxtend`



In [22]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [19]:
# Discretize the features
discretizer2 = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
X_discretized2 = discretizer2.fit_transform(X)


In [27]:
# Convert the discretized data to a DataFrame for rule mining
X_df = pd.DataFrame(X_discretized2, columns=data.feature_names)
X_df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,0.0,1.0,0.0,0.0
1,0.0,1.0,0.0,0.0
2,0.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0
4,0.0,1.0,0.0,0.0


In [26]:
# Apply Apriori algorithm to generate frequent itemsets
frequent_itemsets = apriori(X_df.astype(bool), min_support=0.1, use_colnames=True)
print(frequent_itemsets)

    support                                           itemsets
0      0.60                                (sepal length (cm))
1      0.72                                 (sepal width (cm))
2      0.50                                (petal length (cm))
3      0.50                                 (petal width (cm))
4      0.37              (sepal length (cm), sepal width (cm))
5      0.46             (sepal length (cm), petal length (cm))
6      0.46              (sepal length (cm), petal width (cm))
7      0.23              (petal length (cm), sepal width (cm))
8      0.23               (petal width (cm), sepal width (cm))
9      0.50              (petal length (cm), petal width (cm))
10     0.23  (sepal length (cm), petal length (cm), sepal w...
11     0.23  (sepal length (cm), petal width (cm), sepal wi...
12     0.46  (sepal length (cm), petal length (cm), petal w...
13     0.23  (petal length (cm), petal width (cm), sepal wi...
14     0.23  (sepal length (cm), petal length (cm), pet

In [31]:
# Generate association rules from the frequent itemsets
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print(rules.head(1))

           antecedents         consequents  antecedent support  \
0  (sepal length (cm))  (sepal width (cm))                 0.6   

   consequent support  support  confidence      lift  leverage  conviction  \
0                0.72     0.37    0.616667  0.856481    -0.062    0.730435   

   zhangs_metric  
0      -0.295238  


In [34]:
# Select rules based on Bayesian criteria (simplified)
# In a full implementation, you'd apply a Bayesian model here
selected_rules = rules[(rules['confidence'] > 0.7) & (rules['lift'] > 1.0)]
print(f"selected_rules: {len(selected_rules)} out of {len(rules)} rules")

selected_rules: 21 out of 31 rules


In [36]:
# Build a decision list (ordered set of rules)
rule_list = []
for _, rule in selected_rules.iterrows():
    antecedents = rule['antecedents']
    rule_list.append((antecedents, rule['confidence']))

print(rule_list[:3])

[(frozenset({'sepal length (cm)'}), 0.7666666666666667), (frozenset({'petal length (cm)'}), 0.92), (frozenset({'sepal length (cm)'}), 0.7666666666666667)]


In [37]:
# Function to predict based on rule list
def predict_rule_list(X, rule_list):
    predictions = []
    for _, row in X.iterrows():
        prediction = 0  # Default prediction if no rule applies
        for antecedents, confidence in rule_list:
            if all(row[col] == 1 for col in antecedents):
                prediction = 1  # Rule applies, predict positive class
                break
        predictions.append(prediction)
    return np.array(predictions)

# Apply the rule list to make predictions
X_df_test = pd.DataFrame(X_discretized, columns=data.feature_names)
y_pred = predict_rule_list(X_df_test, rule_list)

# Evaluate the predictions
accuracy = np.mean(y_pred == y)
print(f"Accuracy: {accuracy}")

Accuracy: 0.97
